[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_04_q_learning.ipynb)

# S09 · 04 — Aprendizaje por refuerzo: Q-learning en el hotel

Sprint 9 · M3 (UD 3.12, introducción al aprendizaje por refuerzo)

**Contexto TurisData.** En el aprendizaje supervisado alguien nos da la respuesta correcta. Aquí **no**: un **agente** prueba acciones, recibe **recompensas** (o castigos) y aprende por ensayo y error una **política** (qué hacer en cada situación). Vamos a entrenar a un "botones virtual" que debe llevar a un huésped desde recepción hasta la habitación con vistas al mar, evitando la zona en obras. Es un juguete, pero contiene las piezas de cualquier problema de refuerzo (robots, juegos, precios dinámicos).

**Este cuaderno tiene 4 ejercicios y un mini-reto.** El ejercicio 1 es el mismo que el del cuaderno «en clase» (allí, el 1).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="white")
print("Listo ✔")

### 👀 A2 · EJEMPLO — El entorno del hotel y tres pasos de prueba

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué estado y qué recompensa devuelve el primer paso a la derecha?
2. En el tercer paso (abajo), ¿por qué el estado no cambia y aun así se paga −1?
3. ¿Qué recompensa total tiene llegar a la meta sin pasar por las obras en 7 pasos?

In [ ]:
class EntornoHotel:
    """Cuadrícula sencilla con la misma interfaz que Gymnasium: reset() y step(accion)."""
    FILAS, COLUMNAS = 4, 5
    N_ESTADOS, N_ACCIONES = 20, 4
    PAREDES = {(1, 1), (1, 2), (3, 1)}
    OBRAS = (1, 4)
    META = (3, 4)
    MOVIMIENTOS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
    NOMBRES_ACCION = ["arriba", "abajo", "izquierda", "derecha"]

    def __init__(self, coste_paso=1.0):
        self.coste_paso = coste_paso      # lo que "cuesta" cada paso (1.0 por defecto)

    def reset(self):
        self.pos = (0, 0)
        return self.estado()

    def estado(self):
        return self.pos[0] * self.COLUMNAS + self.pos[1]

    def step(self, accion):
        df, dc = self.MOVIMIENTOS[accion]
        nueva = (self.pos[0] + df, self.pos[1] + dc)
        dentro = 0 <= nueva[0] < self.FILAS and 0 <= nueva[1] < self.COLUMNAS
        if dentro and nueva not in self.PAREDES:
            self.pos = nueva
        recompensa = -self.coste_paso
        if self.pos == self.OBRAS:
            recompensa -= 5.0
        terminado = self.pos == self.META
        if terminado:
            recompensa += 10.0
        return self.estado(), recompensa, terminado


entorno = EntornoHotel()
s = entorno.reset()
print("Estado inicial:", s)
for a in (3, 3, 1):   # derecha, derecha, abajo (choca con la pared)
    s, r, fin = entorno.step(a)
    print(f"acción {EntornoHotel.NOMBRES_ACCION[a]:>9} → estado {s:>2}, recompensa {r:+.0f}, fin={fin}")

### ✏️ A3 · EJERCICIO 1 de 4 — La regla de actualización de Q

**Tu turno:** Escribe `actualizar_q(Q, s, a, r, s2, alpha, gamma)` que aplique la regla **modificando `Q` en su sitio** (no hace falta devolver nada).

**Debe salir:** ✔ Ejercicio 1 correcto (con `Q[0,3]=0`, `r=-1`, `max Q[5]=4`, `alpha=0.5` y `gamma=0.9` sale `Q[0,3] = 1.3`)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Qué valor queda en `Q[0, 3]` y cómo sale?
2. ¿Qué pasaría con `alpha = 1`?
3. ¿Qué pasaría con `gamma = 0`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
def actualizar_q(Q, s, a, r, s2, alpha, gamma):
    # TODO: Primero calcula lo que ahora crees que vale la acción (recompensa + futuro descontado); luego mueve Q[s, a] una fracción alpha hacia ese valor
    ...

Qprueba = np.zeros((20, 4))
Qprueba[5, 3] = 4.0
actualizar_q(Qprueba, s=0, a=3, r=-1.0, s2=5, alpha=0.5, gamma=0.9)
assert abs(Qprueba[0, 3] - 1.3) < 1e-9, "Con Q[0,3]=0, r=-1, max Q[5]=4, alpha=0.5, gamma=0.9 debe salir 0.5·(-1+0.9·4)=1.3"
assert np.count_nonzero(Qprueba) == 2, "Solo debes modificar Q[s, a]"
print("✔ Ejercicio 1 correcto")

### ✏️ A4 · EJERCICIO 2 de 4 — La función epsilon-greedy

**Tu turno:** Escribe `elegir_accion(Q, s, epsilon, rng)`: con probabilidad `epsilon` devuelve una acción aleatoria (`rng.integers(4)`), y si no, la de mayor valor en `Q[s]` (`np.argmax`). Usa `rng.random()` para decidir.

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Qué hace el `if rng.random() < epsilon`?
2. Con `epsilon = 0,2`, ¿qué fracción de las acciones serán aleatorias?
3. ¿Por qué se pasa `rng` a la función en lugar de usar `np.random` directamente?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
def elegir_accion(Q, s, epsilon, rng):
    # TODO: Un único if: si el número aleatorio es menor que epsilon, explora; en otro caso, elige la mejor acción de la fila s
    ...

rng_t = np.random.default_rng(0)
Qt = np.zeros((20, 4)); Qt[7] = [0, 0, 5, 0]
assert all(elegir_accion(Qt, 7, 0.0, rng_t) == 2 for _ in range(50)), "Con epsilon=0 debe elegir siempre la mejor acción (2)"
distintas = {elegir_accion(Qt, 7, 1.0, rng_t) for _ in range(200)}
assert distintas == {0, 1, 2, 3}, "Con epsilon=1 debe explorar las 4 acciones al azar"
print("✔ Ejercicio 2 correcto")

### 👀 A5 · EJEMPLO — Entrenar al agente con tus dos funciones

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A5).

1. ¿Cuál es la recompensa media al principio y al final del entrenamiento?
2. ¿Por qué la recompensa de los últimos episodios (0,9) no llega a +3, el máximo posible?
3. ¿Qué guarda la tabla `Q` al final?

In [ ]:
def entrenar(episodios=1500, alpha=0.2, gamma=0.95, epsilon=0.2, semilla=SEMILLA, max_pasos=50, coste_paso=1.0):
    rng = np.random.default_rng(semilla)
    env = EntornoHotel(coste_paso)
    Q = np.zeros((env.N_ESTADOS, env.N_ACCIONES))
    recompensas = []
    for _ in range(episodios):
        s = env.reset()
        total = 0.0
        for _ in range(max_pasos):
            a = elegir_accion(Q, s, epsilon, rng)
            s2, r, fin = env.step(a)
            actualizar_q(Q, s, a, r, s2, alpha, gamma)
            total += r
            s = s2
            if fin:
                break
        recompensas.append(total)
    return Q, np.array(recompensas)

Q, recompensas = entrenar()
print("Recompensa media de los primeros 50 episodios:", recompensas[:50].mean().round(1))
print("Recompensa media de los últimos 50 episodios: ", recompensas[-50:].mean().round(1))

### 👀 A6 · EJEMPLO — La curva de aprendizaje

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Cuántos episodios tarda aproximadamente en dejar de perder recompensa?
2. ¿Por qué la línea suave se mantiene sobre 0–2 y no sigue subiendo?
3. ¿Por qué se dibuja la media móvil además de cada episodio?

In [ ]:
suave = pd.Series(recompensas).rolling(30).mean()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(recompensas, alpha=0.25, label="cada episodio")
ax.plot(suave, color="tab:red", label="media móvil (30)")
ax.set_xlabel("Episodio"); ax.set_ylabel("Recompensa total")
ax.set_title("El agente mejora con la experiencia"); ax.legend()
sns.despine(); plt.show()

### ✏️ A7 · EJERCICIO 3 de 4 — La política y el camino aprendido

**Tu turno:** Guarda en `politica` un vector de 20 acciones (la mejor en cada estado). Después escribe el `camino` que sigue el agente **sin explorar** desde recepción: lista de estados visitados, empezando en 0, hasta llegar a la meta (estado 19) o hasta un máximo de 30 pasos. Usa `EntornoHotel` y `politica`.

**Debe salir:** `camino de 7 pasos: [0, 1, 2, 3, 8, 13, 14, 19]` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Qué camino sigue el agente y cuántos pasos tiene?
2. ¿Por qué `np.argmax(Q, axis=1)` y no `axis=0`?
3. ¿Por qué el camino evita las obras (estado 9) aunque está justo al lado del camino?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: la mejor acción de cada fila de Q (argmax por filas); para el camino, repite step con la acción que dicta la política hasta llegar a la meta
politica = ...
camino = ...

assert len(politica) == 20, "politica debe tener una acción por estado (20)"
assert camino[0] == 0 and camino[-1] == 19, "El camino debe empezar en 0 y llegar a la habitación (estado 19)"
assert 9 not in camino, "El camino no debería pasar por la zona en obras (estado 9)"
assert len(camino) <= 10, "El camino aprendido debería ser corto (7-9 pasos)"
print(f"camino de {len(camino) - 1} pasos:", camino)
print("✔ Ejercicio 3 correcto")

### 👀 A8 · EJEMPLO — Dibujar la política: flechas y valor de cada casilla

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Qué valor tiene la casilla de recepción (la de arriba a la izquierda) y cuál la de al lado de la meta?
2. ¿Por qué crece el valor al acercarse a la meta?
3. ¿Qué indica la flecha de la casilla de las obras?

In [ ]:
# Dibujo de la política: flechas sobre la cuadrícula
flechas = {0: "↑", 1: "↓", 2: "←", 3: "→"}
mapa = np.full((4, 5), "", dtype=object)
fig, ax = plt.subplots(figsize=(6, 4.5))
valores = Q.max(axis=1).reshape(4, 5)
sns.heatmap(valores, annot=False, cmap="YlGn", cbar_kws={"label": "valor del estado (max Q)"}, ax=ax, linewidths=1, linecolor="white")
for f in range(4):
    for c in range(5):
        if (f, c) in EntornoHotel.PAREDES:
            texto = "PARED"
        elif (f, c) == EntornoHotel.META:
            texto = "META"
        else:
            texto = flechas[politica[f * 5 + c]] + ("\nOBRAS" if (f, c) == EntornoHotel.OBRAS else "")
        ax.text(c + 0.5, f + 0.5, texto, ha="center", va="center", fontsize=13)
ax.set_title("Política aprendida y valor de cada casilla"); ax.set_xticks([]); ax.set_yticks([])
plt.show()

### ✏️ A9 · EJERCICIO 4 de 4 — Con pasos gratis, ¿ayuda explorar?

**Tu turno:** Entrena dos agentes con `entrenar(..., coste_paso=0.0)`: uno con `epsilon=0.0` (nunca explora) y otro con `epsilon=0.3`. Guarda sus recompensas por episodio en `rec_eps0` y `rec_eps03`. Calcula `media_final_eps0` y `media_final_eps03` como la media de sus **últimos 100 episodios** y guarda en `explorar_ayuda` (`True`/`False`) si el agente que explora consigue mayor recompensa media final que el que no.

**Debe salir:** `Sin explorar: 0.0 · Explorando (0,3): 9.1`, `Con gamma=0 …: -49.5` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Qué recompensa final tiene el agente que no explora y cuál el que explora?
2. ¿Por qué el agente que no explora saca exactamente 0?
3. ¿Qué demuestra el experimento sobre exploración y explotación?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
# pista: entrenar devuelve dos cosas (Q, recompensas): guarda la segunda; la media de los últimos 100 son las últimas posiciones del vector
rec_eps0 = ...
rec_eps03 = ...
media_final_eps0 = ...
media_final_eps03 = ...
explorar_ayuda = ...

assert len(rec_eps0) == 1500 and len(rec_eps03) == 1500, "Cada entrenamiento tiene 1500 episodios"
assert explorar_ayuda is True, "Con pasos gratis, el agente que no explora no debería llegar nunca a la meta: revisa coste_paso=0.0 y epsilon"
print(f"Sin explorar: {media_final_eps0:.1f} · Explorando (0,3): {media_final_eps03:.1f}")
_, rec_gamma0 = entrenar(gamma=0.0)
print("Con gamma=0 (sin mirar al futuro), recompensa final:", rec_gamma0[-100:].mean().round(1), "(compárala con el +3 óptimo)")
print("✔ Ejercicio 4 correcto")

### ✏️ A10 · MINI-RETO — Obras a −20: cambiar la recompensa

**Tu turno:** El hotel decide que **pasar por la zona en obras cuesta -20** (reclamaciones de huéspedes). Copia la clase `EntornoHotel` con otro nombre, cambia esa penalización, reentrena con `entrenar` (tendrás que pasar el nuevo entorno: añade un parámetro `clase_entorno`) y comprueba que la política **sigue evitando** las obras. Después responde: ¿qué pasaría si la penalización fuera **+5** (una "recompensa" por pasar)? Es un ejemplo de cómo un mal diseño de la **recompensa** produce comportamientos no deseados.

**Debe salir:** `Camino con obras a -20: [0, 1, 2, 3, 8, 13, 14, 19]` y ✔ Mini-reto correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A10).

1. ¿Qué camino sigue el agente con las obras a −20?
2. ¿Qué pasaría si la penalización fuera +5 (una «recompensa» por pasar)?
3. ¿Qué enseña este experimento sobre diseñar la recompensa?

In [ ]:
# pista: copia la clase con otro nombre (o hereda de EntornoHotel y resta 15 más al pisar las obras) y redefine entrenar con un parámetro nuevo clase_entorno=EntornoHotel
class EntornoObras20(EntornoHotel):
    ...

def entrenar(episodios=1500, alpha=0.2, gamma=0.95, epsilon=0.2, semilla=SEMILLA, max_pasos=50, coste_paso=1.0, clase_entorno=EntornoHotel):
    ...

e = EntornoObras20(); e.reset(); e.pos = (1, 3)
_, r_obras, _ = e.step(3)               # derecha: pisa las obras (fila 1, columna 4)
assert abs(r_obras - (-21.0)) < 1e-9, "Pisar las obras debe costar -20 más el -1 del paso (-21 en total)"
Q20, _ = entrenar(clase_entorno=EntornoObras20)
pol20 = np.argmax(Q20, axis=1)
env20 = EntornoObras20(); s = env20.reset(); camino20 = [s]
for _ in range(30):
    s, _, fin = env20.step(pol20[s])
    camino20.append(s)
    if fin:
        break
assert camino20[-1] == 19 and 9 not in camino20, "La política debe llegar a la habitación evitando las obras (estado 9)"
print("Camino con obras a -20:", camino20)
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga las herramientas: `numpy`, `pandas`, `matplotlib` y `seaborn` (todo el cuaderno está hecho con numpy, sin librerías de refuerzo).
    - `dato(...)` es la función de descarga de datos del curso (en este cuaderno no hace falta, no hay datos externos).
    - Fija la semilla `SEMILLA = 42` para que los entrenamientos den siempre lo mismo.
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. El entorno: el pasillo del hotel

#### A2 · Ejemplo — El entorno del hotel y tres pasos de prueba

**Contexto TurisData.** En el aprendizaje supervisado alguien nos da la respuesta correcta. Aquí **no**: un **agente** prueba acciones, recibe **recompensas** (o castigos) y aprende por ensayo y error una **política** (qué hacer en cada situación). Vamos a entrenar a un «botones virtual» que debe llevar a un huésped desde recepción hasta la habitación con vistas al mar, evitando la zona en obras.

Una cuadrícula de 4 filas × 5 columnas:

```
  S  .  .  .  .        S = recepción (inicio)
  .  #  #  .  X        # = pared (no se puede cruzar)
  .  .  .  .  .        X = zona en obras (recompensa -5, pero se puede pisar)
  .  #  .  .  G        G = habitación con vistas al mar (recompensa +10, fin del episodio)
```

- **Estado:** la casilla donde está el agente (20 posibles, numeradas de 0 a 19 por filas).
- **Acciones:** 0 = arriba, 1 = abajo, 2 = izquierda, 3 = derecha.
- **Recompensa:** -1 por cada paso (el huésped tiene prisa), -5 extra en obras, +10 al llegar a G.
- Si el agente choca con una pared o el borde, **se queda donde está** (y paga el paso).
- **Episodio:** una partida, desde recepción hasta llegar a G (o hasta 50 pasos).

- **Qué hace:**
    - La clase `EntornoHotel` es el mundo del agente, con la misma interfaz que Gymnasium: `reset()` lo coloca en recepción y devuelve el estado inicial; `step(accion)` mueve al agente y devuelve `(estado nuevo, recompensa, ¿terminado?)`.
    - Si el movimiento sale de la cuadrícula o choca con una pared, el agente **se queda donde está** (y paga el paso). Pisar las obras cuesta −5 extra y llegar a la meta da +10.
    - El código final crea un entorno y prueba tres acciones: derecha, derecha y abajo (esta última choca con una pared).
- **Qué debe salir:**
    - `Estado inicial: 0` y tres líneas: `acción derecha → estado 1, recompensa -1, fin=False`, `acción derecha → estado 2, recompensa -1, fin=False` y `acción abajo → estado 2, recompensa -1, fin=False`.
- **Cómo leerlo:**
    - La tercera línea muestra el choque: el estado sigue siendo el 2 y aun así cuesta −1.
- **Si falla:**
    - `SyntaxError`/`IndentationError` → se ha copiado mal la clase: ejecútala tal cual está.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué estado y qué recompensa devuelve el primer paso a la derecha?* → Estado **1** y recompensa **−1** (el coste de cada paso).
2. *En el tercer paso (abajo), ¿por qué el estado no cambia y aun así se paga −1?* → Porque abajo hay una **pared**: el agente se queda donde está, pero el paso cuesta igual.
3. *¿Qué recompensa total tiene llegar a la meta sin pasar por las obras en 7 pasos?* → −1 × 7 + 10 = **+3**: es el máximo posible (el «+3 óptimo»).

</details>

## 2. La tabla Q y su regla de actualización

#### A3 · Ejercicio 1 — La regla de actualización de Q

**Idea.** Guardamos una tabla `Q[estado, acción]` con *"cuánta recompensa total espero si estoy aquí y
hago esta acción y luego sigo lo mejor posible"*. Empieza en ceros. Tras cada paso se corrige un poco:

`Q[s,a] ← Q[s,a] + alpha · ( r + gamma · max(Q[s',:]) − Q[s,a] )`

- `alpha` (0–1): **velocidad de aprendizaje**. Alto = se cree lo último; bajo = cambia despacio.
- `gamma` (0–1): cuánto **importa el futuro**. Con 0 el agente solo mira la recompensa inmediata.
- `r + gamma·max(Q[s'])` es lo que **ahora** creemos que vale esa acción; se compara con lo que creíamos antes.

- **Qué hace:**
    - `objetivo = r + gamma * Q[s2].max()` es lo que **ahora** creemos que vale la acción: la recompensa inmediata más el valor descontado de la mejor opción desde el estado siguiente.
    - `Q[s, a] += alpha * (objetivo - Q[s, a])` mueve el valor antiguo una fracción `alpha` hacia el objetivo. No devuelve nada: modifica `Q` en su sitio.
    - La comprobación crea una tabla de prueba con `Q[5, 3] = 4`, aplica la regla con `alpha=0.5` y `gamma=0.9` y verifica `Q[0, 3] = 0.5 · (−1 + 0.9 · 4) = 1.3` y que solo cambia esa casilla.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `Con Q[0,3]=0, r=-1, max Q[5]=4, alpha=0.5, gamma=0.9 debe salir 0.5·(-1+0.9·4)=1.3` → la fórmula no es la de la regla (¿falta `alpha`, el `max` de `Q[s2]` o el `- Q[s, a]`?).
    - `Solo debes modificar Q[s, a]` → has cambiado otras casillas de `Q` (por ejemplo, la fila entera).
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valor queda en `Q[0, 3]` y cómo sale?* → **1,3**: objetivo = −1 + 0,9 × 4 = 2,6 y se avanza la mitad (`alpha = 0,5`) desde 0: 0,5 × 2,6.
2. *¿Qué pasaría con `alpha = 1`?* → Que `Q[s, a]` se quedaría **directamente** con el objetivo (2,6): se cree lo último que ha visto, sin memoria de lo anterior.
3. *¿Qué pasaría con `gamma = 0`?* → El objetivo sería solo la recompensa inmediata (−1): el agente **no mira al futuro** y no puede aprender a llegar a una meta lejana.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def actualizar_q(Q, s, a, r, s2, alpha, gamma):
    objetivo = r + gamma * Q[s2].max()
    Q[s, a] += alpha * (objetivo - Q[s, a])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. Explorar o explotar: epsilon-greedy

#### A4 · Ejercicio 2 — La función epsilon-greedy

**Idea.** Si el agente siempre hace lo que cree mejor, nunca descubre caminos mejores. Si siempre
experimenta, nunca aprovecha lo aprendido. **Epsilon-greedy**: con probabilidad `epsilon` hace una acción
**al azar** (explorar) y con probabilidad `1 - epsilon` la **mejor conocida** (explotar).

- **Qué hace:**
    - `rng.random() < epsilon` es verdadero con probabilidad `epsilon`: entonces explora y devuelve `int(rng.integers(Q.shape[1]))`, una acción al azar (de 0 a 3).
    - En caso contrario, explota: `int(np.argmax(Q[s]))` es la acción de mayor valor en el estado `s`.
    - La comprobación fija `Qt[7] = [0, 0, 5, 0]`: con `epsilon = 0` debe salir siempre la acción 2, y con `epsilon = 1` deben aparecer las cuatro.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`.
- **Si falla:**
    - `Con epsilon=0 debe elegir siempre la mejor acción (2)` → siempre explora (¿has invertido la comparación `<`?) o no usas `np.argmax(Q[s])`.
    - `Con epsilon=1 debe explorar las 4 acciones al azar` → nunca explora, o la acción aleatoria no cubre las 4 (usa `rng.integers(4)`).
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué hace el `if rng.random() < epsilon`?* → Decide **si explorar**: con probabilidad `epsilon` se hace una acción al azar y con `1 − epsilon` la mejor conocida.
2. *Con `epsilon = 0,2`, ¿qué fracción de las acciones serán aleatorias?* → Aproximadamente el **20 %**.
3. *¿Por qué se pasa `rng` a la función en lugar de usar `np.random` directamente?* → Para que sea **reproducible**: con el mismo generador y semilla se repiten exactamente las mismas decisiones.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def elegir_accion(Q, s, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    return int(np.argmax(Q[s]))
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Entrenar al agente

#### A5 · Ejemplo — Entrenar al agente con tus dos funciones

Un bucle: en cada episodio, el agente empieza en recepción, elige acciones, recibe recompensas y
actualiza `Q` con **tus** dos funciones. Guardamos la recompensa total de cada episodio para ver si mejora.

- **Qué hace:**
    - `entrenar(...)` repite 1 500 episodios: el agente empieza en recepción (`env.reset()`), y en cada paso elige una acción (`elegir_accion`), la ejecuta (`env.step`) y actualiza la tabla (`actualizar_q`), hasta llegar a la meta o agotar 50 pasos.
    - Guarda la recompensa total de cada episodio (`recompensas`) y devuelve la tabla `Q` y esas recompensas.
    - `Q, recompensas = entrenar()` entrena con los valores por defecto (`alpha=0.2`, `gamma=0.95`, `epsilon=0.2`); los dos `print` comparan los primeros y los últimos 50 episodios.
- **Qué debe salir:**
    - `Recompensa media de los primeros 50 episodios: -8.0` y `Recompensa media de los últimos 50 episodios:  0.9`.
- **Cómo leerlo:**
    - La recompensa media pasa de −8,0 a 0,9: el agente ha aprendido. No llega a +3 (el óptimo) porque con `epsilon = 0,2` sigue dando pasos al azar.
- **Si falla:**
    - `NameError: name 'actualizar_q' is not defined` (o `elegir_accion`) → falta el ejercicio 1 (o el ejemplo «ya hecho» de epsilon-greedy).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuál es la recompensa media al principio y al final del entrenamiento?* → **−8,0** en los primeros 50 episodios y **0,9** en los últimos 50.
2. *¿Por qué la recompensa de los últimos episodios (0,9) no llega a +3, el máximo posible?* → Porque con `epsilon = 0,2` el 20 % de las acciones siguen siendo al azar durante el entrenamiento: a veces se pierde algún paso o se pisan las obras.
3. *¿Qué guarda la tabla `Q` al final?* → Para cada estado y acción, la **recompensa total esperada** si se hace esa acción y luego se sigue lo mejor posible.

</details>

#### A6 · Ejemplo — La curva de aprendizaje

- **Qué hace:**
    - `pd.Series(recompensas).rolling(30).mean()` calcula la media móvil de 30 episodios (`suave`).
    - El gráfico dibuja la recompensa de cada episodio (línea tenue, muy ruidosa) y la media móvil (línea roja).
- **Qué debe salir:**
    - Una curva de aprendizaje: la media móvil empieza en torno a −13 en los primeros episodios, sube a ≈ 0 en unas decenas de episodios y después se mantiene entre 0 y 2 con altibajos.
- **Cómo leerlo:**
    - Lo importante es que **sube deprisa y se estabiliza**. Los picos negativos aislados son episodios en los que la exploración al azar lleva al agente a pisar las obras o a dar muchas vueltas.
- **Si falla:**
    - `NameError: name 'recompensas' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos episodios tarda aproximadamente en dejar de perder recompensa?* → Unos **30–60 episodios**: la media móvil pasa de ≈ −13 a ≈ 0.
2. *¿Por qué la línea suave se mantiene sobre 0–2 y no sigue subiendo?* → Porque el agente ya conoce el camino casi óptimo (+3); lo que queda es el efecto de explorar con `epsilon = 0,2`.
3. *¿Por qué se dibuja la media móvil además de cada episodio?* → Porque cada episodio es **muy ruidoso** (depende del azar); la media móvil deja ver la tendencia.

</details>

## 5. Leer la política aprendida

#### A7 · Ejercicio 3 — La política y el camino aprendido

La **política** es la acción de mayor valor en cada estado: `argmax` de cada fila de `Q`.

- **Qué hace:**
    - `np.argmax(Q, axis=1)` da, para cada fila (estado) de `Q`, la acción de mayor valor: es la política (`politica`, 20 acciones).
    - Se crea un entorno nuevo, se coloca al agente en recepción (`reset()`) y se guarda el `camino` (lista de estados).
    - En cada paso se hace la acción que dicta la política (`politica[s]`, **sin explorar**) y se añade el nuevo estado; el bucle acaba al llegar a la meta (`fin`) o a los 30 pasos.
- **Qué debe salir:**
    - `camino de 7 pasos: [0, 1, 2, 3, 8, 13, 14, 19]` y `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - El camino recorre la primera fila, baja por la columna 3 y entra por abajo en la meta (19), **sin pasar por el estado 9** (las obras). 7 pasos es lo más corto posible desde recepción.
- **Si falla:**
    - `politica debe tener una acción por estado (20)` → has usado `axis=0` (devuelve 5 valores) o no has aplicado `np.argmax` a `Q`.
    - `El camino debe empezar en 0 y llegar a la habitación (estado 19)` → `camino` no empieza en el estado inicial o el bucle acaba antes de llegar (falta seguir la política hasta `fin`).
    - `El camino no debería pasar por la zona en obras (estado 9)` / `…debería ser corto` → `Q` no es la de un buen entrenamiento (¿has vuelto a entrenar con otro `epsilon`?).
    - `NameError: name 'Q' is not defined` → falta el ejemplo de entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué camino sigue el agente y cuántos pasos tiene?* → **[0, 1, 2, 3, 8, 13, 14, 19]**: 7 pasos.
2. *¿Por qué `np.argmax(Q, axis=1)` y no `axis=0`?* → Porque queremos **una acción por estado** (cada fila de `Q`); con `axis=0` se obtendría una acción por columna.
3. *¿Por qué el camino evita las obras (estado 9) aunque está justo al lado del camino?* → Porque pisarlas cuesta −5 extra: en `Q`, las acciones que llevan a las obras valen menos que las que bajan por la columna 3.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
politica = np.argmax(Q, axis=1)
entorno = EntornoHotel()
s = entorno.reset()
camino = [s]
for _ in range(30):
    s, _, fin = entorno.step(politica[s])
    camino.append(s)
    if fin:
        break
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A8 · Ejemplo — Dibujar la política: flechas y valor de cada casilla

- **Qué hace:**
    - `Q.max(axis=1).reshape(4, 5)` toma el valor del mejor movimiento de cada estado y lo coloca en la cuadrícula del hotel.
    - `sns.heatmap(...)` pinta la cuadrícula (más verde = más valor) y el bucle escribe encima una flecha con la acción de la política (`↑ ↓ ← →`), «PARED» en las paredes y «META» en la habitación; en la casilla de las obras se añade «OBRAS».
- **Qué debe salir:**
    - Una cuadrícula de 4 × 5 con flechas. Primera fila: `→ → → ↓ ←`; las casillas más verdes están junto a la meta (valor ≈ 9) y las menos verdes son las de la izquierda (recepción ≈ 1,3 y la de abajo a la izquierda ≈ 0,9).
- **Cómo leerlo:**
    - Se lee siguiendo las flechas desde recepción (arriba a la izquierda): el valor sube hacia la meta. La flecha de la casilla de las obras (al final de la segunda fila) apunta hacia abajo.
- **Si falla:**
    - `NameError: name 'politica' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valor tiene la casilla de recepción (la de arriba a la izquierda) y cuál la de al lado de la meta?* → Recepción: **≈ 1,3**; la casilla de al lado de la meta (por ejemplo, la de encima): **≈ 9,0**.
2. *¿Por qué crece el valor al acercarse a la meta?* → Porque quedan **menos pasos** (cada uno cuesta −1) hasta cobrar los +10: el valor de un estado es la recompensa total esperada desde él.
3. *¿Qué indica la flecha de la casilla de las obras?* → Que desde las obras lo mejor es **bajar** hacia la meta (↓): es el camino más corto desde allí.

</details>

## 6. Experimento: ¿qué pasa sin exploración?

#### A9 · Ejercicio 4 — Con pasos gratis, ¿ayuda explorar?

Con el coste de -1 por paso, incluso el agente que nunca explora acaba aprendiendo (cada choque le
resulta "caro" y le empuja a probar otra dirección). Vamos a quitar esa ayuda: **pasos gratis**
(`coste_paso=0`). Ahora la única recompensa llega al final, en la habitación, y hay que *encontrarla*.

- **Qué hace:**
    - `entrenar(epsilon=0.0, coste_paso=0.0)` entrena un agente que **nunca explora** y con pasos gratis: solo recibe recompensa si llega a la meta. `entrenar(epsilon=0.3, ...)` explora el 30 % de las veces. Con `_, rec = ...` nos quedamos solo con las recompensas.
    - `rec[-100:].mean()` es la media de los últimos 100 episodios (`media_final_eps0` y `media_final_eps03`).
    - `explorar_ayuda = bool(media_final_eps03 > media_final_eps0)` es `True` si el que explora gana.
    - La comprobación verifica que ayuda explorar, imprime las dos medias y entrena un tercer agente con `gamma = 0` para comparar.
- **Qué debe salir:**
    - `Sin explorar: 0.0 · Explorando (0,3): 9.1`, `Con gamma=0 (sin mirar al futuro), recompensa final: -49.5 (compárala con el +3 óptimo)` y `✔ Ejercicio 4 correcto`.
- **Cómo leerlo:**
    - Sin explorar y sin coste de pasos, la recompensa final es **0**: el agente no llega nunca a la meta (nunca la ha encontrado, así que no sabe que existe). Explorando, llega y gana unos **9,1**. Con `gamma = 0` (y el coste normal de −1 por paso) el agente no aprende a llegar: gasta los 50 pasos de cada episodio y acumula ≈ −50.
- **Si falla:**
    - `Cada entrenamiento tiene 1500 episodios` → no has guardado la segunda salida de `entrenar` (las recompensas) sino `Q`.
    - `Con pasos gratis, el agente que no explora no debería llegar nunca a la meta: revisa coste_paso=0.0 y epsilon` → `explorar_ayuda` no es `True`: revisa que has puesto `coste_paso=0.0` en los dos entrenamientos y `epsilon=0.0` y `0.3`.
    - `NameError: name 'entrenar' is not defined` → faltan los ejemplos de entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué recompensa final tiene el agente que no explora y cuál el que explora?* → **0,0** y **9,1**.
2. *¿Por qué el agente que no explora saca exactamente 0?* → Porque con pasos gratis y sin explorar **nunca llega a la meta** (empieza con `Q` en ceros y siempre hace la misma acción): no recibe ninguna recompensa.
3. *¿Qué demuestra el experimento sobre exploración y explotación?* → Que **sin explorar no se descubre la recompensa**: hace falta probar acciones nuevas aunque a veces salga mal.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
_, rec_eps0 = entrenar(epsilon=0.0, coste_paso=0.0)
_, rec_eps03 = entrenar(epsilon=0.3, coste_paso=0.0)
media_final_eps0 = rec_eps0[-100:].mean()
media_final_eps03 = rec_eps03[-100:].mean()
explorar_ayuda = bool(media_final_eps03 > media_final_eps0)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 7. Mini-reto: cambiar las reglas del negocio

#### A10 · Mini-reto — Obras a −20: cambiar la recompensa

- **Qué hace:**
    - `EntornoObras20(EntornoHotel)` hereda todo el entorno y solo cambia `step`: llama al `step` original (`super().step(accion)`) y, si el agente está en las obras, resta 15 más a la recompensa (el −5 que ya había + 15 = −20).
    - `entrenar` se redefine con un parámetro más, `clase_entorno=EntornoHotel`, y crea el entorno con `env = clase_entorno(coste_paso)`: así se puede entrenar con cualquier entorno.
    - La comprobación coloca al agente junto a las obras, pisa y verifica que la recompensa es −21 (−20 y el −1 del paso); entrena con el entorno nuevo, lee la política y sigue el camino sin explorar: debe llegar a la meta sin pasar por el estado 9.
- **Qué debe salir:**
    - `Camino con obras a -20: [0, 1, 2, 3, 8, 13, 14, 19]` y `✔ Mini-reto correcto`.
- **Cómo leerlo:**
    - La política sigue siendo el mismo camino de 7 pasos: con −5 ya evitaba las obras, y con −20 las evita con más motivo.
- **Si falla:**
    - `Pisar las obras debe costar -20 más el -1 del paso (-21 en total)` → la penalización no es −20 (¿has restado 20 **además** del −5 existente, que da −25?).
    - `La política debe llegar a la habitación evitando las obras (estado 9)` → `entrenar` sigue usando `EntornoHotel` (falta `env = clase_entorno(coste_paso)`).
    - `TypeError: cannot unpack non-iterable NoneType object` → `entrenar` no devuelve `Q, np.array(recompensas)`.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué camino sigue el agente con las obras a −20?* → El mismo: **[0, 1, 2, 3, 8, 13, 14, 19]**, que no pasa por las obras (estado 9).
2. *¿Qué pasaría si la penalización fuera +5 (una «recompensa» por pasar)?* → Probado: el agente aprende a **ir a las obras y quedarse allí** (choca contra el borde y sigue sumando +4 por paso) y **no llega nunca a la meta**; su recompensa media es ≈ 128 frente a ≈ 1,4 del caso normal.
3. *¿Qué enseña este experimento sobre diseñar la recompensa?* → Que el agente hace lo que se le **premia**, no lo que queremos: si el diseño de la recompensa es malo, aprende un comportamiento absurdo que maximiza el premio.

</details>

## Preguntas de reflexión
1. ¿Qué papel juega `gamma` en el aprendizaje? ¿Por qué con `gamma = 0` no encuentra la meta?
2. ¿Qué ocurre si `epsilon` no baja nunca? ¿Cómo se usaría en producción un agente ya entrenado?
3. Una tabla Q funciona con 20 estados. ¿Y con millones de estados (precios, ocupación, competencia,
   fecha...)? *(pista: redes neuronales → "Deep Q-learning", Sprint 10)*.
4. Un agente que ajusta **precios dinámicos** maximizando ingresos: ¿qué recompensa mal diseñada podría
   perjudicar a los clientes o a la reputación del hotel?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. `gamma` indica **cuánto importa el futuro**: el valor de un estado suma las recompensas futuras multiplicadas por `gamma` en cada paso. Con `gamma = 0` el agente solo mira la recompensa inmediata (−1 por paso) y la meta, que está a varios pasos, nunca «contagia» de valor a las casillas anteriores: no aprende a llegar (en el experimento, recompensa final de −49,5: se pasa los 50 pasos sin llegar).
2. Si `epsilon` no baja nunca, el agente sigue dando pasos al azar siempre (aquí un 20 %) y rinde por debajo de lo que sabe (0,9 frente a +3). En producción se entrena con exploración y, una vez entrenado, se usa **solo la política** (la mejor acción de cada estado, `epsilon = 0`); lo habitual es **ir bajando `epsilon`** durante el entrenamiento.
3. Con millones de estados, la tabla Q **no cabe en memoria** ni se puede visitar cada estado. Se sustituye por una **red neuronal** que estima `Q(s, a)` y generaliza entre estados parecidos: **Deep Q-learning** (Sprint 10).
4. Por ejemplo, si solo premiamos los **ingresos**, el agente puede aprender a subir los precios en los días de máxima demanda hasta niveles abusivos, a cobrar más a quien menos compara o a rechazar reservas rentables a largo plazo: pierde clientes, reputación y puede incurrir en prácticas discriminatorias. La recompensa debe incluir también satisfacción, reclamaciones, ocupación o límites de precio.

</details>

## Qué has aprendido
- Agente, entorno, estado, acción, recompensa y política.
- La regla de Q-learning y los parámetros `alpha` y `gamma`.
- Explorar frente a explotar (epsilon-greedy).
- El diseño de la **recompensa** es la parte delicada: el agente hace lo que se le premia, no lo que queremos.